# Runtime + basemap vector OSM seluruh Indonesia → GCS
Aktifkan LocatioIntelligent dan pilih image/accelerator sama dengan Area (T4x2). Run All interaktif.
Pembuatan basemap dilakukan di CPU dari PBF GCS, zoom native0–14, overzoom tampilan sampai20. Bukan citra satelit.
Build nasional dapat memerlukan waktu dan ruang disk besar; READY hanya sesudah arsip basemap dan runtime tervalidasi. Jangan hentikan hanya karena progress masih memindai PBF.
Tidak ada inference/counting/BQ. File PBF diunduh dari GCS; jika basemap versi ini sudah READY, arsip nasional dipakai ulang.

In [ ]:
import sys,subprocess,os,json,importlib.metadata
os.environ["LI_RUNTIME_BASE_PACKAGES"]=json.dumps({p:importlib.metadata.version(p) for p in ["numpy","pandas","shapely","pyproj","requests","google-auth"]})
subprocess.run([sys.executable,"-m","pip","install","--quiet","google-cloud-storage","google-crc32c"],check=True)

In [ ]:
%%writefile /kaggle/working/national_basemap.py
"""Build a national OSM vector PMTiles archive from the pinned private GCS PBF."""
from pathlib import Path
import hashlib,json,os,shutil,subprocess

PBF_REF={'object':'sources/osm/indonesia/bb1b60f7dfdbb1f19d33fa1ffb0c546f7e75928126ac3a2c004839a3ff2003bc/indonesia.osm.pbf',
    'generation':1792937275821365,'bytes':1739881462,
    'sha256':'bb1b60f7dfdbb1f19d33fa1ffb0c546f7e75928126ac3a2c004839a3ff2003bc'}
PLANETILER_URL='https://github.com/onthegomap/planetiler/releases/download/v0.10.2/planetiler.jar'
PLANETILER_SHA='f310bd0413e2e4512b27f4046d418664e8e1d3bf31603c2a70e23de06c167e4d'
PROFILE_VERSION='indonesia-osm-basic-v1-z14'
MAXZOOM=14
JAVA_HEAP_GB=8
PROFILE_SOURCE=r'''import com.onthegomap.planetiler.FeatureCollector;
import com.onthegomap.planetiler.Planetiler;
import com.onthegomap.planetiler.Profile;
import com.onthegomap.planetiler.config.Arguments;
import com.onthegomap.planetiler.reader.SourceFeature;
import java.nio.file.Path;

public class IndonesiaBasemap implements Profile {
  public String name() { return "Indonesia OSM Basic"; }
  public String description() { return "OSM vector basemap; mapped features only; analytic counting is independent"; }
  public boolean isOverlay() { return false; }
  public String attribution() { return "© OpenStreetMap contributors, ODbL"; }
  private static void named(FeatureCollector.Feature feature, SourceFeature source) {
    if (source.getTag("name") != null) feature.setAttr("name",source.getTag("name"));
  }
  public void processFeature(SourceFeature source, FeatureCollector features) {
    if (source.canBeLine() && source.hasTag("highway")) {
      String road=String.valueOf(source.getTag("highway"));
      int min=road.matches("^(motorway|trunk)(_link)?$")?5:
        road.matches("^(primary|secondary)(_link)?$")?8:
        road.matches("^tertiary(_link)?$")?10:
        road.matches("^(residential|unclassified|living_street)$")?12:13;
      named(features.line("roads").setAttr("class",road).setZoomRange(min,14),source);
    }
    if (source.canBeLine() && source.hasTag("natural","coastline"))
      features.line("coastline").setZoomRange(0,14);
    if (source.canBeLine() && source.hasTag("waterway"))
      named(features.line("waterways").setAttr("class",source.getTag("waterway")).setZoomRange(9,14),source);
    if (source.canBeLine() && source.hasTag("boundary","administrative"))
      features.line("boundaries").setAttr("class",source.getTag("admin_level")).setZoomRange(5,14);
    if (source.canBePolygon()) {
      if (source.hasTag("building") && !source.hasTag("building","no"))
        named(features.polygon("buildings").setAttr("class",source.getTag("building"))
          .setZoomRange(14,14).setMinPixelSizeAtAllZooms(0).setPixelToleranceAtAllZooms(0),source);
      if (source.hasTag("natural","water") || source.hasTag("landuse","reservoir"))
        named(features.polygon("water").setZoomRange(5,14),source);
      else if (source.hasTag("landuse") || source.hasTag("leisure","park","garden","nature_reserve") || source.hasTag("natural","wood","scrub","grassland")) {
        Object kind=source.getTag("landuse");
        if (kind==null) kind=source.getTag("leisure");
        if (kind==null) kind=source.getTag("natural");
        named(features.polygon("landuse").setAttr("class",kind).setZoomRange(8,14),source);
      }
    }
    if (source.isPoint() && source.hasTag("place") && source.getTag("name")!=null) {
      String place=String.valueOf(source.getTag("place"));
      int min=place.equals("city")?5:place.equals("town")?8:11;
      named(features.point("places").setAttr("class",place).setZoomRange(min,14),source);
    }
    if (source.hasTag("amenity") || source.hasTag("shop") || source.hasTag("office") || source.hasTag("healthcare") || source.hasTag("tourism")) {
      Object kind=source.getTag("amenity");
      if (kind==null) kind=source.getTag("shop");
      if (kind==null) kind=source.getTag("office");
      if (kind==null) kind=source.getTag("healthcare");
      if (kind==null) kind=source.getTag("tourism");
      if (source.isPoint()) named(features.point("pois").setAttr("class",kind).setZoomRange(14,14),source);
      else if (source.canBePolygon()) named(features.pointOnSurface("pois").setAttr("class",kind).setZoomRange(14,14),source);
    }
  }
  public static void main(String[] argv) throws Exception {
    Arguments args=Arguments.fromArgsOrConfigFile(argv);
    Planetiler.create(args).setProfile(new IndonesiaBasemap())
      .addOsmSource("osm",Path.of("indonesia.osm.pbf"))
      .setOutput(args.getString("output","output archive","indonesia.pmtiles")).run();
  }
}
'''


def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()


def inspect_archive(path,maxzoom=MAXZOOM):
    from pmtiles.reader import Reader
    from pmtiles.tile import TileType,Compression
    with Path(path).open('rb') as f:
        def read(offset,length):f.seek(offset);return f.read(length)
        reader=Reader(read);header=reader.header();metadata=reader.metadata()
    expected={'roads','buildings','water','waterways','landuse','places','coastline','boundaries','pois'}
    layers={row['id'] for row in metadata.get('vector_layers',[])}
    if header['tile_type']!=TileType.MVT or header['max_zoom']!=maxzoom or header['addressed_tiles_count']<=0:
        raise RuntimeError('BASEMAP_ARCHIVE_INVALID_OR_EMPTY')
    if not layers or not layers<=expected:raise RuntimeError('BASEMAP_LAYER_SCHEMA_INVALID')
    if header['tile_compression'] not in {Compression.GZIP,Compression.NONE}:raise RuntimeError('BASEMAP_COMPRESSION_UNSUPPORTED')
    return {'minzoom':header['min_zoom'],'maxzoom':header['max_zoom'],'tiles':header['addressed_tiles_count'],
            'layers':sorted(layers),'bounds':[header[k]/1e7 for k in ['min_lon_e7','min_lat_e7','max_lon_e7','max_lat_e7']],
            'metadata':metadata}


def build_archive(pbf,output,jar,work,bounds,maxzoom=MAXZOOM,heap_gb=JAVA_HEAP_GB):
    work=Path(work);work.mkdir(parents=True,exist_ok=True);source=work/'IndonesiaBasemap.java'
    source.write_text(PROFILE_SOURCE)
    command=['java',f'-Xmx{heap_gb}g','-cp',str(jar),str(source),f'osm_path={pbf}',f'output={output}',
        f'bounds={",".join(map(str,bounds))}',f'maxzoom={maxzoom}','minzoom=0','download=false',
        f'threads={min(4,os.cpu_count() or 1)}','nodemap_type=sparsearray','nodemap_storage=mmap','compress_temp=true',
        f'tmpdir={work/"tmp"}']
    (work/'build_command.json').write_text(json.dumps(command,indent=2))
    with (work/'build.log').open('w') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:print(line.rstrip(),flush=True);log.write(line);log.flush()
        if process.wait():raise RuntimeError('NATIONAL_BASEMAP_BUILD_FAILED: inspect build.log; no READY published')
    return inspect_archive(output,maxzoom)


def ensure_basemap(bucket,work):
    import requests,osmium
    work=Path(work);work.mkdir(parents=True,exist_ok=True)
    identity={'source':PBF_REF,'profile':PROFILE_VERSION,'profile_sha256':hashlib.sha256(PROFILE_SOURCE.encode()).hexdigest(),
              'planetiler_sha256':PLANETILER_SHA,'maxzoom':MAXZOOM}
    pointer=bucket.blob('basemaps/indonesia/'+PROFILE_VERSION+'/'+PBF_REF['sha256']+'/manifest.json')
    if pointer.exists():
        saved=json.loads(pointer.download_as_bytes());blob=bucket.blob(saved['object'],generation=saved['generation']);blob.reload()
        if saved.get('status')!='READY' or saved.get('identity')!=identity or saved.get('source')!=PBF_REF or blob.size!=saved['bytes'] or (blob.metadata or {}).get('sha256')!=saved['sha256'] or blob.crc32c!=saved.get('crc32c'):
            raise RuntimeError('EXISTING_BASEMAP_MANIFEST_INVALID')
        print('BASEMAP READY: gs://'+bucket.name+'/'+pointer.name+' (verified cache)',flush=True);return saved
    pbf=work/'indonesia.osm.pbf'
    if not pbf.exists():
        if shutil.disk_usage(work).free<8*1024**3:raise RuntimeError('NATIONAL_BASEMAP_INSUFFICIENT_DISK')
        blob=bucket.blob(PBF_REF['object'],generation=PBF_REF['generation'])
        print('BASEMAP 1/4: download pinned PBF from GCS — '+str(PBF_REF['bytes'])+' bytes',flush=True)
        blob.download_to_filename(str(pbf)+'.partial',checksum='crc32c',timeout=600)
        Path(str(pbf)+'.partial').replace(pbf)
    if pbf.stat().st_size!=PBF_REF['bytes'] or file_sha(pbf)!=PBF_REF['sha256']:raise RuntimeError('NATIONAL_PBF_HASH_MISMATCH')
    print('BASEMAP 1/4: PBF checksum verified',flush=True)
    with osmium.io.Reader(str(pbf)) as reader:
        header=reader.header();b=header.box();stamp=header.get('osmosis_replication_timestamp')
        if not b.valid() or not stamp:raise RuntimeError('NATIONAL_PBF_HEADER_INCOMPLETE')
        bounds=[b.bottom_left.lon,b.bottom_left.lat,b.top_right.lon,b.top_right.lat]
    java=subprocess.run(['java','--version'],capture_output=True,text=True) if shutil.which('java') else None
    import re
    major=re.search(r'(?:openjdk|java)\s+(\d+)',java.stdout if java else '')
    compiler=subprocess.run(['java','--list-modules'],capture_output=True,text=True) if java else None
    if not major or int(major.group(1))<21 or compiler is None or 'jdk.compiler@' not in compiler.stdout:
        subprocess.run(['apt-get','install','-y','-qq','openjdk-21-jdk-headless'],check=True)
    jar=work/'planetiler.jar'
    if not jar.exists():
        temporary_jar=work/'planetiler.downloading.jar'
        with requests.get(PLANETILER_URL,stream=True,timeout=(20,90)) as r:
            r.raise_for_status()
            with temporary_jar.open('wb') as f:
                for chunk in r.iter_content(8*1024*1024):f.write(chunk)
        if file_sha(temporary_jar)!=PLANETILER_SHA:raise RuntimeError('PLANETILER_BINARY_HASH_MISMATCH')
        temporary_jar.replace(jar)
    if file_sha(jar)!=PLANETILER_SHA:raise RuntimeError('PLANETILER_BINARY_HASH_MISMATCH')
    output=work/'indonesia.pmtiles';receipt=work/'archive_receipt.json'
    if receipt.exists():
        old=json.loads(receipt.read_text())
        if old['identity']!=identity or not output.exists() or file_sha(output)!=old['sha256']:raise RuntimeError('BASEMAP_LOCAL_CACHE_CHANGED')
        stats=inspect_archive(output)
    else:
        if output.exists():raise RuntimeError('INCOMPLETE_BASEMAP_OUTPUT: inspect before rebuilding')
        temporary=output.with_name('indonesia.building.pmtiles')
        if temporary.exists():temporary.unlink()
        print('BASEMAP 2/4: build national vector archive; native CPU passes/sort follow',flush=True)
        stats=build_archive(pbf,temporary,jar,work/'build',bounds)
        temporary.replace(output)
        receipt.write_text(json.dumps({'identity':identity,'sha256':file_sha(output)},indent=2))
    if not {'roads','buildings','water','coastline','places'}<=set(stats['layers']):raise RuntimeError('NATIONAL_BASEMAP_REQUIRED_LAYERS_MISSING')
    print('BASEMAP 3/4: archive validated; tiles='+str(stats['tiles'])+'; layers='+','.join(stats['layers']),flush=True)
    digest=file_sha(output);key='basemaps/indonesia/'+PROFILE_VERSION+'/'+digest+'/indonesia.pmtiles'
    blob=bucket.blob(key);blob.metadata={'sha256':digest,'profile':PROFILE_VERSION}
    print('BASEMAP 4/4: persist verified archive in GCS — '+str(output.stat().st_size)+' bytes',flush=True)
    if not blob.exists():blob.upload_from_filename(str(output),if_generation_match=0,checksum='crc32c',timeout=600)
    blob.reload()
    if blob.size!=output.stat().st_size or (blob.metadata or {}).get('sha256')!=digest:raise RuntimeError('BASEMAP_GCS_VERIFICATION_FAILED')
    manifest={'schema':'indonesia_basemap_v1','status':'READY','object':key,'generation':int(blob.generation),
        'bytes':blob.size,'sha256':digest,'crc32c':blob.crc32c,'source':PBF_REF,'source_as_of':stamp,
        'profile':PROFILE_VERSION,'identity':identity,**stats,
        'attribution':'© OpenStreetMap contributors, ODbL','coverage':'Geofabrik Indonesia extract; mapped features only',
        'display_only':True,'satellite_imagery':False,'overzoom':'Native maxzoom14; client overzoom allowed, not extra detail'}
    pointer.upload_from_string(json.dumps(manifest),content_type='application/json',if_generation_match=0,checksum='crc32c')
    print('BASEMAP READY: gs://'+bucket.name+'/'+pointer.name,flush=True);return manifest


In [ ]:
"""Ingestion-only CPU job. External downloads happen here, never in Area engine."""
from pathlib import Path
import base64,hashlib,importlib.metadata,json,mimetypes,os,platform,re,subprocess,sys,sysconfig,tarfile


def fingerprint():
    osinfo=platform.freedesktop_os_release()
    return {'python':f'{sys.version_info.major}.{sys.version_info.minor}',
            'platform':sysconfig.get_platform(),'os':osinfo.get('ID'),'os_version':osinfo.get('VERSION_ID'),
            'retained_packages':json.loads(os.environ['LI_RUNTIME_BASE_PACKAGES']) if 'LI_RUNTIME_BASE_PACKAGES' in os.environ else
                {p:importlib.metadata.version(p) for p in ['numpy','pandas','shapely','pyproj','requests','google-auth']}}


def run():
    import requests
    from kaggle_secrets import UserSecretsClient
    from google.oauth2 import service_account
    from google.cloud import storage
    try:info=json.loads(UserSecretsClient().get_secret('LocatioIntelligent'))
    except Exception as error:raise RuntimeError('KAGGLE_SECRET_UNAVAILABLE: activate Secret and Run All interactively') from None
    creds=service_account.Credentials.from_service_account_info(info)
    bucket=storage.Client(project='locdeproject',credentials=creds).bucket('feeder-location-intelligent')
    env=fingerprint();key=hashlib.sha256(json.dumps(env,sort_keys=True).encode()).hexdigest()[:20]
    root=Path('/kaggle/working/area_runtime_prepare')/key;root.mkdir(parents=True,exist_ok=True)
    wheels=root/'wheels';wheels.mkdir(exist_ok=True);native=root/'native';native.mkdir(exist_ok=True)
    retained=env['retained_packages']
    specs=['duckdb==2.0.0','osmnx==2.1.1','geopandas==1.2.0','google-cloud-bigquery==3.46.2',
           'google-cloud-storage','google-crc32c','folium==0.20.0','osmium==4.3.2','pmtiles==3.8.1','cupy-cuda12x==13.6.0','fastrlock==0.8.3']
    specs += [k+'=='+v for k,v in retained.items()]
    request={'specs':specs,'retained':retained};lock=root/'wheels_lock.json'
    cached=json.loads(lock.read_text()) if lock.exists() else None
    if cached and cached['request']==request:
        for name,digest in cached['files'].items():
            if not (wheels/name).is_file() or hashlib.sha256((wheels/name).read_bytes()).hexdigest()!=digest:raise RuntimeError('WHEEL_CACHE_CHANGED')
    else:
        shutil.rmtree(wheels);wheels.mkdir()
        subprocess.run([sys.executable,'-m','pip','download','--only-binary=:all:','--dest',str(wheels),*specs],check=True)
        lock.write_text(json.dumps({'request':request,'files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in wheels.glob('*.whl')}}))
    (root/'requirements.txt').write_text('\n'.join(specs)+'\n')
    subprocess.run(['apt-get','update','-qq'],check=True)
    roots=['python3-gdal','python3-pyproj','osmium-tool']
    output=subprocess.check_output(['apt-cache','depends','--recurse','--no-recommends','--no-suggests',
        '--no-conflicts','--no-breaks','--no-replaces','--no-enhances',*roots],text=True)
    names=sorted(set(line.strip() for line in output.splitlines() if line and not line[0].isspace()
                     and not line.startswith('<') and re.fullmatch(r'[a-z0-9][a-z0-9+.-]*(?::[a-z0-9]+)?',line)))
    for name in names:
        subprocess.run(['apt-get','download',name],cwd=native,check=True,stdout=subprocess.DEVNULL)
    # Extract to a private root; engine never updates its OS or apt repositories.
    system=root/'system';system.mkdir(exist_ok=True)
    for deb in native.glob('*.deb'):subprocess.run(['dpkg-deb','-x',str(deb),str(system)],check=True)
    import glob
    for link in system.rglob('*'):
        if link.is_symlink() and os.readlink(link).startswith('/'):
            target=system/os.readlink(link).lstrip('/');link.unlink();link.symlink_to(os.path.relpath(target,link.parent))
    paths=sorted(set(str(p.parent) for p in system.rglob('*.so*') if p.is_file()))
    for name,binary in [('core-python','/usr/bin/python3'),('osmium',str(system/'usr/bin/osmium'))]:
        script=root/name
        script.write_text('#!/bin/sh\nR="$(CDPATH= cd -- "$(dirname -- "$0")" && pwd)"\n'+
            'export LD_LIBRARY_PATH="'+':'.join(p.replace(str(root),'$R') for p in paths)+':${LD_LIBRARY_PATH:-}"\n'+
            'export PYTHONPATH="$R/system/usr/lib/python3/dist-packages:${PYTHONPATH:-}"\n'+
            'export PROJ_DATA="$R/system/usr/share/proj"\nexport GDAL_DATA="$R/system/usr/share/gdal"\nexport PROJ_NETWORK=OFF\n'+
            'exec "'+binary.replace(str(root),'$R')+'" "$@"\n')
        script.chmod(0o755)
    subprocess.run([str(root/'core-python'),'-c','from osgeo import gdal;import pyproj;assert int(gdal.VersionInfo("VERSION_NUM"))>=3070000;print(gdal.VersionInfo())'],check=True)
    subprocess.run([str(root/'osmium'),'--version'],check=True)
    # Resolve every CSS image/font to data URI in the ingestion job.
    subprocess.run([sys.executable,'-m','pip','install','--no-index','--find-links',str(wheels),'folium==0.20.0','osmium==4.3.2','pmtiles==3.8.1'],check=True)
    import folium
    from folium.plugins import Fullscreen,MarkerCluster
    from urllib.parse import urljoin
    assets={}
    def fetch(url,css=False):
        if url in assets:return assets[url]
        response=requests.get(url,timeout=60);response.raise_for_status();body=response.text
        body=re.sub(r'(?m)//[#@]\s*source(?:Mapping)?URL=[^\r\n]*','',body)
        body=re.sub(r'/\*[#@]\s*source(?:Mapping)?URL=.*?\*/','',body,flags=re.S)
        if css:
            def inline(match):
                link=match.group(1).strip(' \"\'')
                if link.startswith(('data:','#')):return match.group(0)
                absolute=urljoin(url,link);r=requests.get(absolute,timeout=60);r.raise_for_status()
                mime=r.headers.get('content-type') or mimetypes.guess_type(absolute)[0] or 'application/octet-stream'
                return 'url(data:'+mime+';base64,'+base64.b64encode(r.content).decode()+')'
            body=re.sub(r'url\(([^)]+)\)',inline,body)
            if '@import' in body:raise RuntimeError('Unresolved CSS import')
        assets[url]=body;return body
    for cls in [folium.Map,Fullscreen,MarkerCluster]:
        for _,url in cls.default_js:fetch(url)
        for _,url in cls.default_css:fetch(url,True)
    fetch('https://unpkg.com/leaflet.vectorgrid@1.3.0/dist/Leaflet.VectorGrid.bundled.js')
    (root/'frontend.json').write_text(json.dumps(assets))
    from national_basemap import ensure_basemap
    basemap=ensure_basemap(bucket,root.parent/'national_basemap')
    files={str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in root.rglob('*')
           if p.is_file() and p.name!='bundle.json' and not p.is_symlink() and not p.is_relative_to(native)}
    (root/'bundle.json').write_text(json.dumps({'environment':env,'retained':retained,'files':files},indent=2))
    archive=root.parent/'area-runtime.tar.gz'
    with tarfile.open(archive,'w:gz') as t:
        for p in sorted(root.iterdir()):
            if p!=native:t.add(p,arcname=p.name)
    sha=hashlib.sha256(archive.read_bytes()).hexdigest();object_name='runtime/area/'+key+'/'+sha+'/area-runtime.tar.gz'
    blob=bucket.blob(object_name);blob.metadata={'sha256':sha}
    blob.upload_from_filename(str(archive),if_generation_match=0,checksum='crc32c',timeout=600);blob.reload()
    manifest={'schema':'area_gcs_runtime_v2_with_basemap','environment':env,'object':object_name,'generation':int(blob.generation),
        'sha256':sha,'bytes':archive.stat().st_size,'retained':retained,'basemap':basemap}
    pointer=bucket.blob('runtime/area/'+key+'/manifest.json')
    generation=0
    if pointer.exists():pointer.reload();generation=int(pointer.generation)
    pointer.upload_from_string(json.dumps(manifest),content_type='application/json',if_generation_match=generation)
    (root.parent/'runtime_preparation_receipt.json').write_text(json.dumps(manifest,indent=2))
    print('READY: gs://'+bucket.name+'/'+pointer.name)


if __name__=='__main__':run()
